# Notebook 4: 评估与对比

全面评估加权融合方法 vs 多种基线方法。

**基线**: 直接优化 / 默认参数 / 最佳简单 / 最差简单 / 均匀平均
**消融**: 分布分辨率

In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

PROJECT_ROOT = Path(r"D:\data\PPG_HeartRate\Algorithm\outline-PPGtoHR")
ARTIFACTS_DIR = PROJECT_ROOT / "docs" / "research" / "artifacts"
sys.path.insert(0, str(PROJECT_ROOT / "python" / "src"))

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

## 1. 加载融合结果

In [ ]:
with open(ARTIFACTS_DIR / "fusion_results.pkl", "rb") as f:
    fusion = pickle.load(f)

with open(ARTIFACTS_DIR / "simple_params.pkl", "rb") as f:
    params_data = pickle.load(f)

t = fusion["t_common"]
ref_bpm = fusion["ref_bpm"]
weighted_bpm = fusion["weighted_hr_bpm"]
uniform_bpm = fusion["uniform_hr_bpm"]
single_hr = fusion["single_hr_bpm"]
motion_mask = fusion["motion_mask"]
rest_mask = ~motion_mask
class_names = fusion["class_names"]
proba = fusion["aligned_proba"]
burpee_baseline = params_data["burpee_baseline"]

print(f"时间点: {len(t)}, 运动: {motion_mask.sum()}, 静息: {rest_mask.sum()}")

## 2. 基线方法 AAE 对比表

In [ ]:
def compute_aae(pred, ref, mask=None):
    if mask is not None:
        pred, ref = pred[mask], ref[mask]
    return np.mean(np.abs(pred - ref))


methods = {
    "加权融合": weighted_bpm,
    "均匀平均": uniform_bpm,
}
for cn in class_names:
    if cn in single_hr:
        methods[cn] = single_hr[cn]

rows = []
for name, pred in methods.items():
    rows.append({
        "方法": name,
        "AAE_All": compute_aae(pred, ref_bpm),
        "AAE_Rest": compute_aae(pred, ref_bpm, rest_mask),
        "AAE_Motion": compute_aae(pred, ref_bpm, motion_mask),
    })

if burpee_baseline:
    rows.append({
        "方法": "直接优化*",
        "AAE_All": burpee_baseline.get("min_err_hf", float("nan")),
        "AAE_Rest": float("nan"),
        "AAE_Motion": float("nan"),
    })

df_aae = pd.DataFrame(rows).sort_values("AAE_All")
display(df_aae.round(2))

## 3. 逐窗口误差分布箱线图

In [ ]:
order = df_aae["方法"].tolist()
error_records = []
for name, pred in methods.items():
    errs = np.abs(pred - ref_bpm)
    for i, e in enumerate(errs):
        error_records.append({"方法": name, "绝对误差": e,
                              "阶段": "运动" if motion_mask[i] else "静息"})
df_errors = pd.DataFrame(error_records)

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_errors, x="方法", y="绝对误差", order=order, ax=ax)
ax.set_title("各方法逐窗口绝对误差分布")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, phase in zip(axes, ["静息", "运动"]):
    subset = df_errors[df_errors["阶段"] == phase]
    sns.boxplot(data=subset, x="方法", y="绝对误差", order=order, ax=ax)
    ax.set_title(f"{phase}期间误差")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 4. 统计显著性检验

In [ ]:
weighted_errs = np.abs(weighted_bpm - ref_bpm)
uniform_errs = np.abs(uniform_bpm - ref_bpm)

best_single_name = min(single_hr, key=lambda k: compute_aae(single_hr[k], ref_bpm))
best_single_errs = np.abs(single_hr[best_single_name] - ref_bpm)

print("统计检验 (配对 Wilcoxon):")
print("=" * 60)

_, p1 = stats.wilcoxon(weighted_errs, uniform_errs)
d1 = (np.mean(weighted_errs) - np.mean(uniform_errs)) / np.std(weighted_errs - uniform_errs)
print(f"加权 vs 均匀:  p={p1:.4f}, Cohen's d={d1:.3f}")

_, p2 = stats.wilcoxon(weighted_errs, best_single_errs)
d2 = (np.mean(weighted_errs) - np.mean(best_single_errs)) / np.std(weighted_errs - best_single_errs)
print(f"加权 vs 最佳单参({best_single_name}): p={p2:.4f}, Cohen's d={d2:.3f}")
print("\n注: Cohen's d < 0 表示加权融合误差更小")

## 5. 误差改进与分类器分布的关系

In [ ]:
improvement = uniform_errs - weighted_errs

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                          gridspec_kw={"height_ratios": [2, 1]})

ax = axes[0]
ax.stackplot(range(len(t)),
             *[proba[:, i] for i in range(proba.shape[1])],
             labels=class_names, alpha=0.8)
ax.set_ylabel("概率")
ax.set_title("分类器分布 & 误差改进")
ax.legend(loc="upper right", fontsize=8)

ax = axes[1]
colors = ["green" if v > 0 else "red" for v in improvement]
ax.bar(range(len(t)), improvement, color=colors, alpha=0.6, width=1.0)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("窗口索引")
ax.set_ylabel("误差改进 (BPM)")
ax.set_title("加权融合 vs 均匀平均 (正值=加权更好)")
plt.tight_layout()
plt.show()

## 6. 消融: 分布分辨率

In [ ]:
global_proba = proba.mean(axis=0, keepdims=True)
global_proba = np.repeat(global_proba, len(t), axis=0)

hr_matrix = np.zeros((len(class_names), len(t)))
for i, cn in enumerate(class_names):
    hr_matrix[i] = single_hr.get(cn, ref_bpm)

global_weighted_bpm = np.sum(global_proba * hr_matrix, axis=0)

print("消融: 分布分辨率")
print("-" * 40)
print(f"窗口级分布: AAE={compute_aae(weighted_bpm, ref_bpm):.2f}")
print(f"全局分布:   AAE={compute_aae(global_weighted_bpm, ref_bpm):.2f}")
print(f"均匀平均:   AAE={compute_aae(uniform_bpm, ref_bpm):.2f}")

## 7. 汇总报告

In [ ]:
print("=" * 60)
print("研究总结")
print("=" * 60)
print(f"\n数据: {fusion.get('burpee_csv', 'N/A')}")
print(f"窗口: {len(t)} (运动 {motion_mask.sum()}, 静息 {rest_mask.sum()})")
print(f"\n加权融合 AAE: {fusion['aae_weighted']['all']:.2f} BPM")
print(f"均匀平均 AAE: {fusion['aae_uniform']['all']:.2f} BPM")

if burpee_baseline:
    baseline_err = burpee_baseline.get("min_err_hf", 0)
    weighted_err = fusion['aae_weighted']['all']
    if baseline_err > 0:
        pct = (baseline_err - weighted_err) / baseline_err * 100
        print(f"直接优化 AAE: {baseline_err:.2f} BPM")
        print(f"相对直接优化: {'+'if pct > 0 else ''}{pct:.1f}%")

print(f"\n最佳单参数: {best_single_name} ({compute_aae(single_hr[best_single_name], ref_bpm):.2f} BPM)")
better = compute_aae(weighted_bpm, ref_bpm) < compute_aae(uniform_bpm, ref_bpm)
print(f"分类器是否带来改进: {'是' if better else '否'}")